# Moduł 09. Python Unit Tests — Pokrycie kodu — v2.0

Autor: Kamil Bartocha

## Rozkład jazdy

- 🔹 1. Metryki pokrycia kodu - line vs branch
- 🔹 2. `coverage.py` i `pytest-cov` - uruchamianie i raporty
- 🔹 3. Konfiguracja i dobre praktyki pokrycia

## 1. 🔹 Metryki pokrycia kodu - line vs branch

Pokrycie kodu (code coverage) mierzy jaka część kodu źródłowego
jest wykonywana przez testy. Dwie główne metryki:

**Line coverage** - ile linii kodu zostało wykonanych:

```
Linie wykonane / Wszystkie linie × 100%
```

**Branch coverage** - ile gałęzi decyzji zostało sprawdzonych.
Każdy `if/else`, pętla, `try/except` tworzy dwie gałęzie.

Przykład: funkcja z jednym `if` ma dwie gałęzie (True i False).
Test tylko dla True daje 100% line coverage (linia `if` wykonana),
ale tylko 50% branch coverage (gałąź False pominięta):

```python
def is_adult(age):       # linia 1
    if age >= 18:        # linia 2 - branch: True i False
        return True      # linia 3
    return False         # linia 4

# Test: is_adult(20) == True
# Line coverage: 75% (linia 4 nie wykonana)
# Branch coverage: 50% (gałąź age < 18 nie sprawdzona)
```

Branch coverage jest silniejszą metryką - wykrywa niepokryte
ścieżki logiczne, które line coverage pomija.

In [ ]:
import sys, subprocess, tempfile, os

def _run(cmd, cwd=None):
    result = subprocess.run(
        cmd, capture_output=True, text=True, cwd=cwd
    )
    output = result.stdout + result.stderr
    print(output[-2000:])

# Demonstracja: ten sam kod, line vs branch coverage
src = '''
def classify_age(age):
    if age < 0:
        return 'invalid'
    elif age < 18:
        return 'minor'
    elif age < 65:
        return 'adult'
    else:
        return 'senior'
'''

test_partial = '''
from src_age import classify_age

def test_adult():
    assert classify_age(30) == 'adult'
'''

test_full = '''
from src_age import classify_age

def test_adult():   assert classify_age(30) == 'adult'
def test_minor():   assert classify_age(10) == 'minor'
def test_senior():  assert classify_age(70) == 'senior'
def test_invalid(): assert classify_age(-1) == 'invalid'
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_age.py'), 'w') as f:
        f.write(src)

    print('=== Częściowe testy (tylko adult) ===')
    with open(os.path.join(tmpdir, 'test_age.py'), 'w') as f:
        f.write(test_partial)
    _run([sys.executable, '-m', 'coverage', 'run',
          '--branch', '-m', 'pytest', 'test_age.py', '-q'], cwd=tmpdir)
    _run([sys.executable, '-m', 'coverage', 'report',
          '--include=src_age.py', '-m'], cwd=tmpdir)

    print('=== Pełne testy (wszystkie gałęzie) ===')
    with open(os.path.join(tmpdir, 'test_age.py'), 'w') as f:
        f.write(test_full)
    _run([sys.executable, '-m', 'coverage', 'run',
          '--branch', '-m', 'pytest', 'test_age.py', '-q'], cwd=tmpdir)
    _run([sys.executable, '-m', 'coverage', 'report',
          '--include=src_age.py', '-m'], cwd=tmpdir)

---

### 🐍 Ćwiczenia — metryki pokrycia

1. Uruchom `coverage run -m pytest` i `coverage report -m`
8. Zinterpretuj raport: skąd wynika różnica między line a branch
   coverage

In [ ]:
# Ćwiczenie 1: coverage run + coverage report
# Uruchom testy dla klasy BankAccount przez coverage
# i wypisz raport z zaznaczeniem niepokrytych linii (-m).

src = '''
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('amount must be positive')
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('amount must be positive')
        if amount > self.balance:
            raise ValueError('insufficient funds')
        self.balance -= amount

    def get_balance(self):
        return self.balance
'''

test_partial = '''
from src_bank import BankAccount

def test_deposit():
    acc = BankAccount()
    acc.deposit(100)
    assert acc.get_balance() == 100

def test_withdraw():
    acc = BankAccount(100)
    acc.withdraw(30)
    assert acc.get_balance() == 70
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_bank.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_bank.py'), 'w') as f:
        f.write(test_partial)

    # TODO: uzupełnij polecenie coverage run
    _run([sys.executable, '-m', 'coverage', 'run',
          '-m', 'pytest', ...], cwd=tmpdir)
    _run([sys.executable, '-m', 'coverage', 'report',
          '--include=src_bank.py', ...], cwd=tmpdir)

In [ ]:
# Ćwiczenie 8: różnica line vs branch coverage
# Poniższy kod ma 100% line coverage po podanych testach,
# ale niepełne branch coverage. Uruchom z --branch i wyjaśnij wynik.

src = '''
def process_order(order):
    total = order['quantity'] * order['price']
    if order.get('discount'):
        total = total * (1 - order['discount'])
    if total > 1000:
        shipping = 0
    else:
        shipping = 20
    return total + shipping
'''

test_code = '''
from src_order import process_order

def test_with_discount_large():
    order = {"quantity": 5, "price": 300, "discount": 0.1}
    result = process_order(order)
    assert result == 5 * 300 * 0.9  # 1350, free shipping

def test_no_discount_small():
    order = {"quantity": 1, "price": 50}
    result = process_order(order)
    assert result == 50 + 20
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_order.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_order.py'), 'w') as f:
        f.write(test_code)

    print('=== Bez --branch (line coverage) ===')
    _run([sys.executable, '-m', 'coverage', 'run',
          '-m', 'pytest', 'test_order.py', '-q'], cwd=tmpdir)
    _run([sys.executable, '-m', 'coverage', 'report',
          '--include=src_order.py', '-m'], cwd=tmpdir)

    print('=== Z --branch (branch coverage) ===')
    _run([sys.executable, '-m', 'coverage', 'run',
          '--branch', '-m', 'pytest', 'test_order.py', '-q'], cwd=tmpdir)
    _run([sys.executable, '-m', 'coverage', 'report',
          '--include=src_order.py', '-m'], cwd=tmpdir)

# Która gałąź nie jest pokryta? Dodaj test pokrywający ją:
# TODO: uzupełnij brakujący przypadek testowy

## 2. 🔹 `coverage.py` i `pytest-cov` - uruchamianie i raporty

Dwa sposoby uruchamiania pomiaru pokrycia:

**Przez `coverage` CLI:**

```bash
coverage run -m pytest tests/         # zbiera dane
coverage report -m                    # raport w terminalu
coverage html                         # raport HTML w htmlcov/
coverage xml                          # raport XML (dla CI)
coverage erase                        # czyści .coverage
```

**Przez `pytest-cov` plugin:**

```bash
pytest --cov=src tests/                        # pokrycie modułu src/
pytest --cov=. --cov-report=term-missing       # z linijkami w terminalu
pytest --cov=. --cov-report=html               # raport HTML
pytest --cov=. --cov-fail-under=80             # fail jeśli < 80%
```

Flaga `--cov-report=term-missing` pokazuje numery niepokrytych linii
bezpośrednio w tabeli wyników - najwygodniejsza forma podczas
lokalnego rozwoju.

Plik `.coverage` przechowuje dane zebrane przez `coverage run`.
Przy kolejnym uruchomieniu dane są nadpisywane (lub łączone
przy `coverage run --append`).

In [ ]:
# Przykład: pytest-cov --cov-report=term-missing
src = '''
class Calculator:
    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b

    def multiply(self, a, b):
        return a * b

    def divide(self, a, b):
        if b == 0:
            raise ZeroDivisionError("cannot divide by zero")
        return a / b
'''

test_code = '''
from src_calc import Calculator

def test_add():
    assert Calculator().add(2, 3) == 5

def test_subtract():
    assert Calculator().subtract(10, 4) == 6
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_calc.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_calc.py'), 'w') as f:
        f.write(test_code)

    print('=== pytest-cov: term-missing ===')
    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_calc',
        '--cov-report=term-missing',
        '-p', 'no:cacheprovider',
        'test_calc.py', '-q'
    ], cwd=tmpdir)

---

### 🐍 Ćwiczenia — coverage.py i pytest-cov

4. Uruchom pytest z `--cov=src --cov-report=term-missing`
2. Wygeneruj raport HTML i zidentyfikuj niepokryte linie
7. Napisz testy zwiększające pokrycie od 60% do 90%

In [ ]:
# Ćwiczenie 4: pytest-cov z --cov-report=term-missing
# Uruchom testy dla UserValidator przez pytest-cov.
# Uzupełnij flagę --cov= i odczytaj które linie nie są pokryte.

src = '''
class UserValidator:
    def validate(self, user):
        errors = []
        if not user.get('name'):
            errors.append('name is required')
        if not user.get('email') or '@' not in user['email']:
            errors.append('valid email is required')
        age = user.get('age')
        if age is None:
            errors.append('age is required')
        elif age < 0 or age > 150:
            errors.append('age out of range')
        return errors
'''

test_code = '''
from src_validator import UserValidator

def test_valid_user():
    v = UserValidator()
    errors = v.validate({"name": "Alice", "email": "alice@x.com", "age": 30})
    assert errors == []

def test_missing_name():
    v = UserValidator()
    errors = v.validate({"email": "bob@x.com", "age": 25})
    assert "name is required" in errors
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_validator.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_validator.py'), 'w') as f:
        f.write(test_code)

    # TODO: uzupełnij --cov= i --cov-report=
    _run([
        sys.executable, '-m', 'pytest',
        '--cov=...',
        '--cov-report=...',
        '-p', 'no:cacheprovider',
        'test_validator.py', '-q'
    ], cwd=tmpdir)

In [ ]:
# Ćwiczenie 2: raport HTML
# Wygeneruj raport HTML dla poniższego kodu.
# Po uruchomieniu komórki otwórz plik htmlcov/index.html
# i zidentyfikuj niepokryte linie (zaznaczone na czerwono).

src = '''
class OrderProcessor:
    def process(self, order):
        if order["status"] == "new":
            return self._handle_new(order)
        elif order["status"] == "pending":
            return self._handle_pending(order)
        elif order["status"] == "cancelled":
            return self._handle_cancelled(order)
        else:
            raise ValueError(f"unknown status: {order[\"status\"]}")

    def _handle_new(self, order):
        return {"action": "created", "id": order["id"]}

    def _handle_pending(self, order):
        return {"action": "waiting", "id": order["id"]}

    def _handle_cancelled(self, order):
        return {"action": "refunded", "id": order["id"]}
'''

test_code = '''
from src_orders import OrderProcessor

def test_new_order():
    proc = OrderProcessor()
    result = proc.process({"status": "new", "id": 1})
    assert result["action"] == "created"
'''

import shutil
html_dir = os.path.join(os.getcwd(), 'htmlcov_ex2')

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_orders.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_orders.py'), 'w') as f:
        f.write(test_code)

    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_orders',
        f'--cov-report=html:{html_dir}',
        '-p', 'no:cacheprovider',
        'test_orders.py', '-q'
    ], cwd=tmpdir)

print(f'Raport HTML: {html_dir}/index.html')
print('Otwórz plik w przeglądarce aby zobaczyć niepokryte linie.')

In [ ]:
# Ćwiczenie 7: zwiększenie pokrycia od ~60% do 90%
# Poniższy kod ma tylko 2 testy pokrywające część metod.
# Dopisz testy tak by pokrycie wzrosło do co najmniej 90%.

src = '''
class Inventory:
    def __init__(self):
        self._items = {}

    def add(self, name, quantity):
        if quantity <= 0:
            raise ValueError("quantity must be positive")
        self._items[name] = self._items.get(name, 0) + quantity

    def remove(self, name, quantity):
        if name not in self._items:
            raise KeyError(f"{name} not in inventory")
        if quantity > self._items[name]:
            raise ValueError("insufficient stock")
        self._items[name] -= quantity
        if self._items[name] == 0:
            del self._items[name]

    def count(self, name):
        return self._items.get(name, 0)

    def total_items(self):
        return sum(self._items.values())
'''

test_code = '''
from src_inventory import Inventory

def test_add():
    inv = Inventory()
    inv.add("apple", 5)
    assert inv.count("apple") == 5

def test_total():
    inv = Inventory()
    inv.add("apple", 3)
    inv.add("banana", 2)
    assert inv.total_items() == 5

# TODO: dopisz testy pokrywające:
#   - add z ujemna iloscia (ValueError)
#   - remove poprawne
#   - remove nieistniejacego (KeyError)
#   - remove z niewystarczajacym stanem (ValueError)
#   - remove do zera (kasowanie klucza)
#   - count nieistniejacego (zwraca 0)
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_inventory.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_inventory.py'), 'w') as f:
        f.write(test_code)

    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_inventory',
        '--cov-report=term-missing',
        '-p', 'no:cacheprovider',
        'test_inventory.py', '-q'
    ], cwd=tmpdir)

## 3. 🔹 Konfiguracja i dobre praktyki pokrycia

Plik `.coveragerc` (lub sekcja `[coverage]` w `setup.cfg` /
`pyproject.toml`) pozwala centralnie konfigurować pomiar pokrycia:

```ini
# .coveragerc
[run]
branch = True
source = src
omit =
    */migrations/*
    */settings*.py
    */tests/*

[report]
fail_under = 85
show_missing = True
exclude_lines =
    pragma: no cover
    if __name__ == .__main__.:
```

Komentarz `# pragma: no cover` wyklucza linię lub blok z raportu.
Używamy go dla kodu który nie wymaga testowania:

```python
def debug_dump(obj):  # pragma: no cover
    import pprint
    pprint.pprint(vars(obj))
```

Dobre praktyki:

- 80-90% pokrycia to rozsądny cel produkcyjny
- 100% pokrycia nie gwarantuje braku błędów
- branch coverage ważniejszy niż line coverage
- nie pokrywaj `if __name__ == '__main__':`, konfiguracji, migracji
- `--cov-fail-under` w CI blokuje merge gdy pokrycie spada

In [ ]:
# Przykład: pragma: no cover + .coveragerc
src = '''
import logging

logger = logging.getLogger(__name__)

class DataPipeline:
    def run(self, data):
        cleaned = self._clean(data)
        return self._transform(cleaned)

    def _clean(self, data):
        return [x for x in data if x is not None]

    def _transform(self, data):
        return [x * 2 for x in data]

    def debug_state(self):  # pragma: no cover
        logger.debug("pipeline debug: %s items", 0)

if __name__ == "__main__":  # pragma: no cover
    pipeline = DataPipeline()
    print(pipeline.run([1, None, 3]))
'''

coveragerc = '''
[run]
branch = True

[report]
show_missing = True
exclude_lines =
    pragma: no cover
    if __name__ == .__main__.
'''

test_code = '''
from src_pipeline import DataPipeline

def test_run_filters_none():
    p = DataPipeline()
    assert p.run([1, None, 3]) == [2, 6]

def test_run_empty():
    p = DataPipeline()
    assert p.run([]) == []
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_pipeline.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_pipeline.py'), 'w') as f:
        f.write(test_code)
    with open(os.path.join(tmpdir, '.coveragerc'), 'w') as f:
        f.write(coveragerc)

    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_pipeline',
        '--cov-report=term-missing',
        '-p', 'no:cacheprovider',
        'test_pipeline.py', '-q'
    ], cwd=tmpdir)

---

### 🐍 Ćwiczenia — konfiguracja i praktyki

5. Dodaj `# pragma: no cover` do kodu który nie wymaga testów
3. *(Trudniejsze)* Włącz `branch=True` i znajdź niepokryte gałęzie
6. *(Trudniejsze)* Skonfiguruj `.coveragerc` z `omit`, `include`,
   progiem minimalnym
9. *(Trudniejsze)* Skonfiguruj `--cov-fail-under=85` i podłącz
   do CI (plik `Makefile`)

In [ ]:
# Ćwiczenie 5: pragma: no cover
# Klasa Config ma metodę debug_print używaną tylko developersko.
# Dodaj pragma: no cover do metody i bloku __main__,
# a następnie sprawdź że raport pokazuje 100% pokrycia.

src = '''
class Config:
    def __init__(self, data):
        self._data = data

    def get(self, key, default=None):
        return self._data.get(key, default)

    def debug_print(self):  # TODO: dodaj pragma: no cover
        for k, v in self._data.items():
            print(f"  {k} = {v}")

if __name__ == "__main__":  # TODO: dodaj pragma: no cover
    cfg = Config({"host": "localhost"})
    cfg.debug_print()
'''

test_code = '''
from src_config import Config

def test_get_existing():
    cfg = Config({"host": "localhost", "port": 5432})
    assert cfg.get("host") == "localhost"

def test_get_default():
    cfg = Config({})
    assert cfg.get("missing", "default_val") == "default_val"
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_config.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_config.py'), 'w') as f:
        f.write(test_code)

    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_config',
        '--cov-report=term-missing',
        '-p', 'no:cacheprovider',
        'test_config.py', '-q'
    ], cwd=tmpdir)
# Czy pokrycie wynosi 100%? Jeśli nie - dodaj pragma: no cover.

In [ ]:
# Ćwiczenie 3: branch=True - znajdź niepokryte gałęzie *(Trudniejsze)*
# Uruchom coverage z --branch dla poniższego kodu.
# Zidentyfikuj i opisz brakujące gałęzie na podstawie raportu.
# Następnie dopisz testy pokrywające brakujące gałęzie.

src = '''
def shipping_cost(weight, express=False, member=False):
    if weight <= 0:
        raise ValueError("weight must be positive")
    base = weight * 2.5
    if express:
        base *= 2
    if member:
        base *= 0.9
    return round(base, 2)
'''

test_code = '''
from src_shipping import shipping_cost

def test_standard():
    assert shipping_cost(10) == 25.0

def test_express():
    assert shipping_cost(10, express=True) == 50.0

# TODO: dopisz testy pokrywające brakujące gałęzie:
# - ujemna waga
# - member discount
# - express + member
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_shipping.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_shipping.py'), 'w') as f:
        f.write(test_code)

    _run([
        sys.executable, '-m', 'coverage', 'run',
        '--branch', '-m', 'pytest', 'test_shipping.py', '-q'
    ], cwd=tmpdir)
    _run([
        sys.executable, '-m', 'coverage', 'report',
        '--include=src_shipping.py', '-m'
    ], cwd=tmpdir)

In [ ]:
# Ćwiczenie 6: .coveragerc z omit i fail_under *(Trudniejsze)*
# Skonfiguruj .coveragerc tak by:
#   - branch = True
#   - omitować plik helpers.py
#   - fail_under = 80
# Sprawdź że raport nie zawiera helpers.py i respektuje próg.

src_main = '''
def greet(name):
    if not name:
        return "Hello, stranger!"
    return f"Hello, {name}!"
'''

src_helpers = '''
# plik pomocniczy - nie wymaga testowania
def _internal_format(msg):
    return f"[LOG] {msg}"
'''

test_code = '''
from src_main import greet

def test_greet_with_name():
    assert greet("Alice") == "Hello, Alice!"

def test_greet_empty():
    assert greet("") == "Hello, stranger!"
'''

# TODO: uzupełnij .coveragerc
coveragerc = '''
[run]
branch = ...

[report]
fail_under = ...
omit =
    ...
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_main.py'), 'w') as f:
        f.write(src_main)
    with open(os.path.join(tmpdir, 'helpers.py'), 'w') as f:
        f.write(src_helpers)
    with open(os.path.join(tmpdir, 'test_main.py'), 'w') as f:
        f.write(test_code)
    with open(os.path.join(tmpdir, '.coveragerc'), 'w') as f:
        f.write(coveragerc)

    _run([
        sys.executable, '-m', 'pytest',
        '--cov=.',
        '--cov-config=.coveragerc',
        '--cov-report=term-missing',
        '-p', 'no:cacheprovider',
        'test_main.py', '-q'
    ], cwd=tmpdir)

In [ ]:
# Ćwiczenie 9: --cov-fail-under + Makefile *(Trudniejsze)*
# Utwórz Makefile z celami: test, coverage, coverage-check.
# coverage-check ma failować gdy pokrycie < 85%.
# Uruchom make coverage-check i zaobserwuj wynik.

src = '''
def fizzbuzz(n):
    if n % 15 == 0:
        return "FizzBuzz"
    elif n % 3 == 0:
        return "Fizz"
    elif n % 5 == 0:
        return "Buzz"
    return str(n)
'''

test_code = '''
from src_fizz import fizzbuzz

def test_fizz():     assert fizzbuzz(3) == "Fizz"
def test_buzz():     assert fizzbuzz(5) == "Buzz"
def test_fizzbuzz(): assert fizzbuzz(15) == "FizzBuzz"
def test_number():   assert fizzbuzz(7) == "7"
'''

makefile = '''
.PHONY: test coverage coverage-check

test:
	python3 -m pytest test_fizz.py -v

coverage:
	python3 -m pytest --cov=src_fizz --cov-report=term-missing test_fizz.py

coverage-check:
	python3 -m pytest --cov=src_fizz --cov-fail-under=85 test_fizz.py -q
'''

with tempfile.TemporaryDirectory() as tmpdir:
    with open(os.path.join(tmpdir, 'src_fizz.py'), 'w') as f:
        f.write(src)
    with open(os.path.join(tmpdir, 'test_fizz.py'), 'w') as f:
        f.write(test_code)
    with open(os.path.join(tmpdir, 'Makefile'), 'w') as f:
        f.write(makefile)

    print('=== make coverage ===')
    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_fizz', '--cov-report=term-missing',
        '-p', 'no:cacheprovider', 'test_fizz.py', '-q'
    ], cwd=tmpdir)

    print('=== make coverage-check (fail-under=85) ===')
    _run([
        sys.executable, '-m', 'pytest',
        '--cov=src_fizz', '--cov-fail-under=85',
        '-p', 'no:cacheprovider', 'test_fizz.py', '-q'
    ], cwd=tmpdir)

    print(f'Makefile zapisany w: {tmpdir}/Makefile')
    with open(os.path.join(tmpdir, 'Makefile')) as f:
        print(f.read())